# Chạy ngầm thử nghiệm E09.15 trên Google Colab (`duytung/01-cub-gom-cum-cosine-gmm`)
**Lưu ý để đóng tab / tắt máy vẫn chạy tiếp (Colab Pro):**
1. Vào **Runtime → Change runtime type** (hoặc **Tools → Settings**) và bật **Background execution** (nếu tài khoản Colab Pro có tùy chọn này).
2. Chạy lần lượt **Cell 1 → Cell 4**. Cell 4 sẽ đẩy toàn bộ pipeline vào tiến trình ngầm (`nohup`) và trả kết quả về Drive (`e0915_bg.log`, `e0915_summary.txt`), nên không bị kẹt ở tab trình duyệt.
3. Khi mở lại máy, chỉ cần mở notebook và chạy **Cell 5** để kiểm tra tiến độ / xem bảng kết quả.

In [ ]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone / cập nhật repo public (giữ nguyên features/ và runs/ đã xong trên Drive)
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
REPO_DIR = f'{WORK_DIR}/VB-CGCD-defect-testing-only'
os.makedirs(WORK_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/DuytungDeveloper/VB-CGCD-defect-testing-only.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin
!git checkout duytung/01-cub-gom-cum-cosine-gmm
!git reset --hard origin/duytung/01-cub-gom-cum-cosine-gmm
!git status -sb

In [ ]:
# 3. Cài gói thiếu, gỡ torchao, và đảm bảo JAX nhận GPU (nếu đang chạy CPU)
!pip uninstall -y -q torchao
!pip install -q continuum numpyro prettytable

import subprocess
has_gpu = subprocess.call(
    ['python', '-c', 'import jax; assert any(d.platform != "cpu" for d in jax.devices())']
) == 0
if not has_gpu:
    print('>>> JAX đang chạy CPU, thử cài jax[cuda12]...')
    !pip install -q -U "jax[cuda12]"

!python -c "import torch, jax, numpyro, continuum, prettytable, sklearn, scipy, datasets, peft; print('Imports OK! Torch CUDA:', torch.cuda.is_available(), '| JAX devices:', jax.devices())"

In [ ]:
# 4. Chạy ngầm theo giai đoạn (mặc định STAGE='auto': tự bỏ qua phần đã xong, resume đúng bước bị dừng)
# Có thể đổi STAGE thành: 'auto' | 'phase1' | 's0' | 's1' | 's2' | 's3' | 's4' | 'checks' | 'e0915' | 'summary'
STAGE = 'auto'

import os, subprocess, time

WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/e0915_bg.log'
PID_FILE = f'{WORK_DIR}/e0915_bg.pid'

running = False
if os.path.exists(PID_FILE):
    old_pid = open(PID_FILE).read().strip()
    if old_pid and os.path.exists(f'/proc/{old_pid}'):
        running = True
        print(f'>>> Pipeline đang chạy ngầm sẵn với PID {old_pid}. Chạy Cell 5 để xem log!')

if not running:
    !chmod +x colab/run_e0915_bg.sh
    proc = subprocess.Popen(
        ['nohup', 'bash', 'colab/run_e0915_bg.sh', STAGE],
        stdout=open(LOG_FILE, 'w'),
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    with open(PID_FILE, 'w') as f:
        f.write(str(proc.pid))
    time.sleep(2)
    print(f'>>> Đã khởi chạy ngầm STAGE={STAGE}, PID={proc.pid}! Log: {LOG_FILE}')
    !head -n 20 {LOG_FILE}

In [ ]:
# 5. Kiểm tra tiến độ & xem kết quả (chạy cell này bất cứ lúc nào)
import os

WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/e0915_bg.log'
PID_FILE = f'{WORK_DIR}/e0915_bg.pid'
SUMMARY_OUT = f'{WORK_DIR}/e0915_summary.txt'

pid = open(PID_FILE).read().strip() if os.path.exists(PID_FILE) else ''
alive = bool(pid and os.path.exists(f'/proc/{pid}'))
print(f'Trạng thái PID={pid}:', 'ĐANG CHẠY' if alive else 'ĐÃ DỪNG / XONG')

if os.path.exists(LOG_FILE):
    print('\n--- 40 dòng log gần nhất ---')
    !tail -n 40 {LOG_FILE}

if os.path.exists(SUMMARY_OUT):
    print('\n--- BẢNG KẾT QUẢ E09.15 ---')
    print(open(SUMMARY_OUT).read())